# AnimeGANv3 — Video to Animation on Colab T4
Select a **T4 GPU**, then **Runtime → Run all**. The notebook clones the benchmark branch, installs dependencies, downloads the Hayao model, benchmarks the CUDA pipeline on 120 frames at three inference resolutions, then renders the full video.


In [ ]:
import os, subprocess, sys
REPO = '/content/video-to-animation'
if os.path.isdir(os.path.join(REPO, '.git')):
    subprocess.run(['git','fetch','origin','animeganv3-baseline'],cwd=REPO,check=True)
    subprocess.run(['git','reset','--hard','origin/animeganv3-baseline'],cwd=REPO,check=True)
else:
    subprocess.run(['git','clone','-b','animeganv3-baseline','https://github.com/v-tech-hub/video-to-animation.git',REPO],check=True)
os.chdir(REPO)
print('Working directory:', os.getcwd())
subprocess.run(['git','log','-1','--oneline'],check=True)
!nvidia-smi
!apt-get -qq update && apt-get -qq install -y ffmpeg
!pip uninstall -y -q onnxruntime onnxruntime-gpu || true
!pip install -q -r requirements-colab.txt


In [ ]:
import sys
import subprocess
import onnxruntime as ort

print('Python:', sys.version)
print('ORT version:', ort.__version__)
print('ORT device:', ort.get_device())
print('Available providers:', ort.get_available_providers())
print('GPU snapshot:')
subprocess.run([
    'nvidia-smi',
    '--query-gpu=name,driver_version,memory.total,memory.used,utilization.gpu',
    '--format=csv,noheader'
], check=False)

assert 'CUDAExecutionProvider' in ort.get_available_providers(), 'CUDAExecutionProvider unavailable.'


In [ ]:
## Benchmark video
By default the notebook asks you to upload your own video (`USE_UPLOAD = True`). Set it to `False` only to use the public sample.


## Benchmark video
By default the notebook downloads a public sample video automatically, so **Run all** needs no upload. Set `USE_UPLOAD = True` in the next cell only when testing your own video.


In [ ]:
# Default reproducible benchmark video from Google's Media CDN.
# Set USE_UPLOAD = True only when you want to test your own video.
USE_UPLOAD = False

from pathlib import Path
import urllib.request
import shutil

if USE_UPLOAD:
    from google.colab import files
    uploaded = files.upload()
    videos = [n for n in uploaded if Path(n).suffix.lower() in {'.mp4','.mov','.avi','.mkv','.webm'}]
    assert videos, 'Upload a video file.'
    src = Path(videos[0])
    VIDEO = '/content/input' + src.suffix.lower()
    shutil.copyfile(src, VIDEO)
else:
    VIDEO = '/content/input.mp4'
    SAMPLE_URL = 'https://storage.googleapis.com/gtv-videos-bucket/sample/ForBiggerBlazes.mp4'
    if not Path(VIDEO).exists():
        print('Downloading sample benchmark video...')
        urllib.request.urlretrieve(SAMPLE_URL, VIDEO)

print('Video:', VIDEO, f'({Path(VIDEO).stat().st_size / 1024 / 1024:.2f} MB)')


In [ ]:
import os, time, cv2
import numpy as np
from PIL import Image
import onnxruntime as ort
from pathlib import Path
import urllib.request

# Self-contained benchmark cell: safe to run even if the model setup cell was skipped.
MODEL = globals().get('MODEL', '/content/AnimeGANv3_Hayao_36.onnx')
MODEL_URL = 'https://github.com/TachibanaYoshino/AnimeGANv3/releases/download/v1.1.0/AnimeGANv3_Hayao_36.onnx'
if not Path(MODEL).exists():
    print('Model missing; downloading AnimeGANv3 Hayao model...')
    urllib.request.urlretrieve(MODEL_URL, MODEL)
print('Benchmark model:', MODEL)

BENCH_FRAMES = 120
RESOLUTIONS = [1280, 1024, 960]  # max edge -> portrait: 720x1280, 576x1024, 536x960 for 9:16

cap = cv2.VideoCapture(VIDEO)
src_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
src_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
raw_frames = []
while len(raw_frames) < BENCH_FRAMES:
    ok, bgr = cap.read()
    if not ok:
        break
    raw_frames.append(bgr)
cap.release()
assert raw_frames, 'Could not read benchmark frames.'
print(f'Benchmark source: {len(raw_frames)} frames, {src_w}x{src_h}')

sess = ort.InferenceSession(MODEL, providers=['CUDAExecutionProvider'])
assert sess.get_providers()[0] == 'CUDAExecutionProvider', sess.get_providers()
input_name = sess.get_inputs()[0].name
print('CUDA providers:', sess.get_providers())

results = []
for max_edge in RESOLUTIONS:
    scale = min(1.0, max_edge / max(src_w, src_h))
    iw = max(256, int(round(src_w * scale)) // 8 * 8)
    ih = max(256, int(round(src_h * scale)) // 8 * 8)
    print(f'\n=== CUDA pipeline {iw}x{ih} ===', flush=True)

    # Preprocess all benchmark frames exactly like the renderer, timed separately.
    prepared = []
    t0 = time.perf_counter()
    for bgr in raw_frames:
        small = cv2.resize(bgr, (iw, ih), interpolation=cv2.INTER_AREA)
        rgb = cv2.cvtColor(small, cv2.COLOR_BGR2RGB).astype(np.float32)
        prepared.append((rgb / 127.5 - 1.0)[None, ...])
    preprocess_s = time.perf_counter() - t0

    # Warm up GPU so initialization does not pollute steady-state timing.
    for f in prepared[:min(10, len(prepared))]:
        sess.run(None, {input_name: f})

    outputs = []
    t0 = time.perf_counter()
    for f in prepared:
        outputs.append(sess.run(None, {input_name: f})[0])
    inference_s = time.perf_counter() - t0

    # Match renderer's output upscale/uint8 conversion.
    post = []
    t0 = time.perf_counter()
    for out in outputs:
        img = (out.squeeze() + 1.0) / 2.0 * 255.0
        img = img.clip(0, 255).astype(np.uint8)
        img = cv2.resize(img, (src_w, src_h), interpolation=cv2.INTER_LINEAR)
        post.append(img)
    postprocess_s = time.perf_counter() - t0

    # Measure the current MJPEG writer, including disk I/O.
    tmp = f'/content/bench_{iw}x{ih}.avi'
    writer = cv2.VideoWriter(tmp, cv2.VideoWriter_fourcc(*'MJPG'), 60.0, (src_w, src_h))
    assert writer.isOpened(), f'Could not open benchmark writer: {tmp}'
    t0 = time.perf_counter()
    for img in post:
        writer.write(img[:, :, ::-1])
    writer.release()
    write_s = time.perf_counter() - t0
    try:
        os.remove(tmp)
    except OSError:
        pass

    total_s = preprocess_s + inference_s + postprocess_s + write_s
    fps = len(raw_frames) / total_s
    inf_fps = len(raw_frames) / inference_s
    row = {
        'size': f'{iw}x{ih}', 'fps': fps, 'inference_fps': inf_fps,
        'preprocess': preprocess_s, 'inference': inference_s,
        'postprocess': postprocess_s, 'write': write_s,
    }
    results.append(row)
    print(f'preprocess : {preprocess_s:7.2f}s  {preprocess_s/len(raw_frames)*1000:7.1f} ms/frame')
    print(f'inference  : {inference_s:7.2f}s  {inference_s/len(raw_frames)*1000:7.1f} ms/frame  ({inf_fps:.2f} FPS)')
    print(f'postprocess: {postprocess_s:7.2f}s  {postprocess_s/len(raw_frames)*1000:7.1f} ms/frame')
    print(f'MJPEG write: {write_s:7.2f}s  {write_s/len(raw_frames)*1000:7.1f} ms/frame')
    print(f'pipeline   : {total_s:7.2f}s  {fps:.2f} FPS')

print('\n=== CUDA RESOLUTION SUMMARY ===')
for r in results:
    print(f"{r['size']:>10} | inference {r['inference_fps']:5.2f} FPS | measured pipeline {r['fps']:5.2f} FPS")

BEST_DEVICE = 'gpu'
RENDER_MAX_EDGE = 1024
print('\nFull render backend: CUDA')
print(f'Full render max edge: {RENDER_MAX_EDGE} (chosen from benchmark sweet spot)')


In [ ]:
import time, sys, threading
os.chdir(REPO)
os.makedirs('output', exist_ok=True)

# Fail before an expensive render if Colab is still using an old checkout.
script = os.path.join(REPO, 'tools', 'video2anime.py')
code = open(script, 'r').read()
assert 'direct ffmpeg H.264 pipe' in code, 'Old render code detected; restart/re-clone first.'

cmd = ['python','-u',script,'-i',VIDEO,'-o',os.path.join(REPO,'output'),'-m',MODEL,'-d',BEST_DEVICE,'--max-edge',str(RENDER_MAX_EDGE)]
print('=== Full AnimeGANv3 render ===', flush=True)
print('Backend:', BEST_DEVICE, flush=True)
print('Command:', ' '.join(cmd), flush=True)

stop_gpu = threading.Event()
def gpu_monitor():
    while not stop_gpu.wait(5):
        q = subprocess.run(['nvidia-smi','--query-gpu=utilization.gpu,memory.used,power.draw','--format=csv,noheader,nounits'], capture_output=True, text=True)
        if q.returncode == 0:
            print('[gpu] util%, memMiB, powerW:', q.stdout.strip(), flush=True)
threading.Thread(target=gpu_monitor, daemon=True).start()

start = time.perf_counter()
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='', flush=True)
returncode = proc.wait()
stop_gpu.set()
elapsed = time.perf_counter() - start
print(f'\nExit code: {returncode}')
print(f'Wall time: {elapsed:.2f} s')
if returncode != 0:
    raise RuntimeError(f'AnimeGANv3 failed with exit code {returncode}')


In [ ]:
import cv2
import subprocess
from pathlib import Path
from google.colab import files

cap = cv2.VideoCapture(VIDEO)
frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)
cap.release()
duration = frames / fps if fps else 0
render_fps = frames / elapsed if elapsed else 0
rtf = elapsed / duration if duration else 0
print(f'Frames: {frames}')
print(f'Source FPS: {fps:.3f}')
print(f'Source duration: {duration:.2f} s')
print(f'Render throughput: {render_fps:.2f} FPS')
print(f'Realtime factor: {rtf:.2f}x')

outs = sorted(Path(REPO, 'output').glob('*.mp4'), key=lambda x: x.stat().st_mtime, reverse=True)
assert outs, 'No rendered MP4 found.'
final_output = outs[0]
probe = subprocess.run([
    'ffprobe','-v','error','-select_streams','v:0',
    '-show_entries','stream=codec_name,width,height,r_frame_rate',
    '-of','default=noprint_wrappers=1',str(final_output)
], text=True, capture_output=True)
print('Output:', final_output)
print(probe.stdout)
assert probe.returncode == 0 and 'codec_name=h264' in probe.stdout, probe.stderr or 'Output is not valid H.264.'
files.download(str(final_output))
